# Remote Cleanup 01: Baseline and Individual Release

Focused manual test for normal remote operation and individual object release. Re-run START 2 before each release/error cell if you want independent MMIO, Buffer, and GPIO checks. The final section uses raw RPC calls only for malformed-handle edge cases that normal PYNQ API cannot create.

In [ ]:
import gc
import json
import os
import numpy as np

REMOTE_IP = os.environ.get("PYNQ_REMOTE_DEVICES", "192.168.2.197").split(",")[0].strip()
os.environ["PYNQ_REMOTE_DEVICES"] = REMOTE_IP
OVERLAY_PATH = "/workspace/phd/PYNQ.remote-dev/applications/PYNQ/tests/resizer.xsa"

import pynq

print("REMOTE_IP:", REMOTE_IP)
print("OVERLAY_PATH:", OVERLAY_PATH)

In [ ]:
# START 1: create a RemoteDevice through normal PYNQ import/probe.
# Re-run this cell when you want constructor auto_cleanup=True to run again.
if hasattr(pynq.Device, "_active_device"):
    delattr(pynq.Device, "_active_device")
if hasattr(pynq.Device, "_devices"):
    delattr(pynq.Device, "_devices")

remote_devices = [d for d in pynq.Device.devices if d.has_capability("REMOTE")]
if not remote_devices:
    raise RuntimeError("No RemoteDevice found. Check PYNQ_REMOTE_DEVICES and board connectivity.")

device = remote_devices[0]
print("device:", device)

In [ ]:
# Baseline: normal remote overlay + DMA path should still work.
device = [d for d in pynq.Device.devices if d.has_capability("REMOTE")][0]
overlay = pynq.Overlay(OVERLAY_PATH, device=device)
dma = overlay.axi_dma_0
resizer = overlay.resize_accel_0
size = 500
fake_img = np.random.randint(0, 256, (size, size, 3), dtype=np.uint8)
in_buffer = device.allocate(shape=(size, size, 3), dtype=np.uint8, cacheable=1)
out_buffer = device.allocate(shape=(size, size, 3), dtype=np.uint8, cacheable=1)
in_buffer[:] = fake_img
resizer.register_map.src_rows = size
resizer.register_map.src_cols = size
resizer.register_map.dst_rows = size
resizer.register_map.dst_cols = size
dma.sendchannel.transfer(in_buffer)
dma.recvchannel.transfer(out_buffer)
resizer.write(0x00, 0x81)
dma.sendchannel.wait()
dma.recvchannel.wait()
assert np.array_equal(in_buffer[:], out_buffer[:])
print("Baseline remote DMA test passed.")

In [ ]:
# START 2: load the overlay and create one MMIO, one buffer, and optional GPIO.
# Re-run START 1 first if you want a fresh device/constructor cleanup.
overlay = pynq.Overlay(OVERLAY_PATH, device=device)
resizer = overlay.resize_accel_0
resizer.read(0)

mmio = resizer.mmio
mmio_id = mmio._remote_map.mmio_id

buffer = device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
buffer[:] = np.arange(16, dtype=np.uint32)
buffer.flush()
buffer_id = buffer.buffer_id

gpio = None
gpio_id = None
gpio_pin = None
gpio_path = None
base_path = pynq.GPIO.get_gpio_base_path(device=device)
npins = pynq.GPIO.get_gpio_npins(device=device)
if base_path and npins:
    gpio_pin = pynq.GPIO.get_gpio_pin(0, device=device)
    gpio_path = f"/sys/class/gpio/gpio{gpio_pin}"
    if device.exists_file(gpio_path).exists:
        print("GPIO already exported, skipping GPIO object:", gpio_path)
    else:
        gpio = pynq.GPIO(gpio_pin, "in", device=device)
        gpio.read()
        gpio_id = gpio._gpio_id
else:
    print("GPIO sysfs base not available; GPIO cells will skip.")

print("MMIO id:  ", mmio_id)
print("Buffer id:", buffer_id)
print("GPIO id:  ", gpio_id, "path:", gpio_path)

## Normal PYNQ API Error Cells

Malformed remote IDs are not normally reachable through the public PYNQ API, because PYNQ receives opaque IDs from the server. This section uses normal PYNQ operations that should fail visibly. The final section imports the `pb2` modules and calls raw RPC stubs for the malformed-ID boundary checks.

## Individual Release Errors

Run START 2 before each of these cells if you want to test each service independently. Each cell releases one live PYNQ object, then immediately uses that same normal PYNQ object again. The cell should error. For exact server-side `Object not found` assertions, use the automated pytest tests.

In [ ]:
# Error test: release one current MMIO, then use that same normal PYNQ object again.
# Re-run START 2 before this cell if mmio was already released or cleaned.
mmio.close()
mmio.read(0)

In [ ]:
# Error test: free one current buffer, then use that same normal PYNQ object again.
# Re-run START 2 before this cell if buffer was already released or cleaned.
buffer.freebuffer()
buffer.physical_address

In [ ]:
# Error test: release one current GPIO, then use that same normal PYNQ object again.
# Re-run START 2 before this cell if gpio was already released or cleaned.
if gpio_id is None:
    print("Skipping GPIO release test because no GPIO object was created.")
else:
    gpio.release()
    gpio.read()

## Non-API Raw RPC Edge Cases

Run Setup or START 1 first so `device` exists. These cells intentionally bypass the public PYNQ API and send malformed handles directly to the gRPC server. Each operation should print `INVALID_ARGUMENT` and an `Invalid ... ID` message. This is useful for checking the server boundary, but it is not normal user-facing PYNQ usage.

In [ ]:
import grpc
from pynq.remote import buffer_pb2, gpio_pb2, mmio_pb2

if "device" not in globals():
    remote_devices = [d for d in pynq.Device.devices if d.has_capability("REMOTE")]
    if not remote_devices:
        raise RuntimeError("No RemoteDevice found. Check PYNQ_REMOTE_DEVICES and board connectivity.")
    device = remote_devices[0]

print("raw RPC checks will use device:", device)

In [ ]:
# Non-API error test: malformed MMIO handle.
try:
    device._stub["mmio"].read(
        mmio_pb2.ReadRequest(
            mmio_id="not-a-valid-mmio-id",
            offset=0,
            length=4,
            word_order="little",
        )
    )
except grpc.RpcError as exc:
    print(exc.code().name, exc.details())
    assert exc.code() == grpc.StatusCode.INVALID_ARGUMENT
    assert "Invalid MMIO ID" in exc.details()
else:
    raise AssertionError("Malformed MMIO handle unexpectedly succeeded")

In [ ]:
# Non-API error test: malformed Buffer handle.
try:
    device._stub["buffer"].physical_address(
        buffer_pb2.AddressRequest(buffer_id="not-a-valid-buffer-id")
    )
except grpc.RpcError as exc:
    print(exc.code().name, exc.details())
    assert exc.code() == grpc.StatusCode.INVALID_ARGUMENT
    assert "Invalid Buffer ID" in exc.details()
else:
    raise AssertionError("Malformed Buffer handle unexpectedly succeeded")

In [ ]:
# Non-API error test: malformed GPIO handle.
try:
    device._stub["gpio"].read(gpio_pb2.GpioReadRequest(gpio_id="not-a-valid-gpio-id"))
except grpc.RpcError as exc:
    print(exc.code().name, exc.details())
    assert exc.code() == grpc.StatusCode.INVALID_ARGUMENT
    assert "Invalid GPIO ID" in exc.details()
else:
    raise AssertionError("Malformed GPIO handle unexpectedly succeeded")